# ER-CyRIS Siklus 1 — STEP 1 ONLY
## Verifikasi 4 Dataset Sebelum Eksperimen

Notebook ini **belum menjalankan S0–S5**.

Tujuan satu-satunya adalah memastikan bahwa empat dataset dimuat dengan benar sebelum eksperimen confirmatory dilanjutkan:

1. CICIDS2018
2. HDFS
3. BGL
4. UNSW-NB15

Perbaikan terhadap notebook confirmatory v2:
- **CICIDS2018:** audit/dedup full feature vector tetap dilakukan karena memang ditemukan duplicate flow identik pada eksperimen awal.
- **HDFS:** TIDAK didedup berdasarkan 8 fitur agregat, karena dua block berbeda boleh memiliki pola agregat yang sama.
- **BGL:** TIDAK didedup berdasarkan engineered features, karena event log berbeda boleh menghasilkan fitur yang sama.
- **UNSW-NB15:** loader mendeteksi apakah file memiliki header. Jika tidak, schema resmi 49 kolom digunakan.

Jangan lanjut ke S0 sebelum output notebook ini diperiksa.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, zlib, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

BASE = '/content/drive/MyDrive/ercyris_siklus2/'

CFG = {
    'CICIDS': BASE + '02-14-2018.csv',
    'HDFS':   BASE + 'HDFS.log',
    'HLABEL': BASE + 'anomaly_label.csv',
    'BGL':    BASE + 'BGL.log',
    'UNSW':   BASE + 'UNSW-NB15_1.csv',
    'SEED': 42,
    'MAX_CICIDS': 500_000,
    'MAX_BGL': 50_000,
}

for k,v in CFG.items():
    if isinstance(v, str):
        print(f"{k:10s}: {'✅' if os.path.exists(v) else '❌'} {v}")

In [ ]:
def clean_cols(cols):
    out = []
    for c in cols:
        c = str(c).strip().lower().replace(' ', '_').replace('/', '_per_')
        c = re.sub(r'[^a-z0-9_]+', '_', c).strip('_')
        out.append(c)
    return out

def stable_hash_1000(text):
    return zlib.crc32(str(text).encode('utf-8')) % 1000

def audit_df(name, X, y):
    print(f"\n{'='*72}")
    print(name)
    print('='*72)
    print(f"rows      : {len(X):,}")
    print(f"features  : {X.shape[1]:,}")
    print(f"normal    : {(np.asarray(y)==0).sum():,}")
    print(f"attack/anom: {(np.asarray(y)==1).sum():,}")
    print(f"positive% : {100*np.mean(y):.4f}%")
    print(f"duplicate feature vectors (AUDIT ONLY): {X.duplicated().sum():,}")
    print("first columns:", list(X.columns[:12]))

In [ ]:
# ============================================================
# 1. CICIDS2018
# ============================================================
def load_cicids_verify():
    df = pd.read_csv(CFG['CICIDS'], low_memory=False)

    if len(df) > CFG['MAX_CICIDS']:
        df = df.sample(n=CFG['MAX_CICIDS'], random_state=CFG['SEED'])

    df.columns = clean_cols(df.columns)
    label_col = next(c for c in df.columns if 'label' in c)

    y_label = df[label_col].astype(str).str.strip().str.upper()
    y = (y_label != 'BENIGN').astype(int)

    drop = [label_col] + [
        c for c in df.columns
        if any(k in c for k in ['timestamp','flow_id','src_ip','dst_ip','source_ip','destination_ip'])
    ]
    X = df.drop(columns=list(set(drop)), errors='ignore').replace([np.inf,-np.inf], np.nan)

    # CICIDS: duplicate flow-feature vectors memang diaudit dan dibuang.
    before = len(X)
    tmp = X.copy()
    tmp['__y__'] = y.values
    feat = list(X.columns)
    dup = tmp.duplicated(subset=feat, keep='first')
    tmp = tmp.loc[~dup].reset_index(drop=True)

    y2 = tmp.pop('__y__').astype(int).values
    X2 = tmp

    print(f"CICIDS dedup: {before:,} → {len(X2):,} "
          f"({before-len(X2):,} duplicate rows removed)")
    return X2, y2

In [ ]:
# ============================================================
# 2. HDFS
# ============================================================
def load_hdfs_verify():
    LP = re.compile(r'^(\d{6})\s+(\d{6})\s+\d+\s+(\w+)\s+([^:]+):\s+(.*)')
    BP = re.compile(r'(blk_-?\d+)')
    SM = {'DEBUG':0,'INFO':1,'WARN':2,'WARNING':2,'ERROR':3,'FATAL':4}

    bd = defaultdict(lambda:{
        'n':0,'e':0,'w':0,'comp':set(),'ms':0,'ss':0
    })

    with open(CFG['HDFS'], 'r', errors='replace') as f:
        for line in tqdm(f, desc='parse HDFS', leave=False):
            m = LP.match(line)
            if not m:
                continue
            _,_,lv,comp,msg = m.groups()
            sev = SM.get(lv.upper(),1)

            for blk in BP.findall(msg):
                d = bd[blk]
                d['n'] += 1
                d['comp'].add(comp.strip())
                d['ms'] = max(d['ms'], sev)
                d['ss'] += sev
                if sev >= 3: d['e'] += 1
                if sev == 2: d['w'] += 1

    labels = pd.read_csv(CFG['HLABEL'])
    lmap = dict(zip(
        labels['BlockId'].astype(str),
        (labels['Label'].astype(str).str.upper() == 'ANOMALY').astype(int)
    ))

    rows, ys = [], []
    for blk,d in bd.items():
        if blk not in lmap:
            continue

        rows.append({
            'n_events': d['n'],
            'n_error': d['e'],
            'n_warn': d['w'],
            'n_comp': len(d['comp']),
            'max_sev': d['ms'],
            'mean_sev': d['ss']/max(d['n'],1),
            'error_ratio': d['e']/max(d['n'],1),
            'warn_ratio': d['w']/max(d['n'],1),
        })
        ys.append(lmap[blk])

    # PENTING: jangan drop_duplicates di sini.
    # BlockId sudah menjadi unit analisis unik sebelum fitur agregat dibentuk.
    X = pd.DataFrame(rows)
    y = np.asarray(ys, dtype=int)
    return X, y

In [ ]:
# ============================================================
# 3. BGL
# ============================================================
def load_bgl_verify():
    SM = {
        'INFO':1,'DEBUG':0,'WARN':2,'WARNING':2,'ERROR':3,'SEVERE':4,
        'FATAL':4,'APPFPO':3,'KERN':2,'KERNSVC':3,'APPFPW':2
    }
    AKW = ['fatal','error','fail','exception','abort','illegal',
           'segfault','panic','crash','out of memory','overflow']

    rows = []

    with open(CFG['BGL'], 'r', errors='replace') as f:
        for seq,line in enumerate(tqdm(f, desc='parse BGL', leave=False)):
            p = line.split(None, 8)
            if len(p) < 9:
                continue

            is_anom = 0 if p[0] == '-' else 1
            sev = SM.get(p[6].upper(),1)
            msg = p[8].lower()

            try:
                hour = int(p[3].split(':')[0]) if ':' in p[3] else 12
            except Exception:
                hour = 12

            rows.append({
                'sev': sev,
                'is_err': int(sev >= 3),
                'is_warn': int(sev == 2),
                'hour': hour,
                'off_hr': int(hour < 6),
                'peak_hr': int(8 <= hour <= 18),
                'err_kw': int(any(k in msg for k in AKW)),
                'has_num': int(bool(re.search(r'\b\d{4,}\b', msg))),
                'msg_len': min(len(msg),500),
                'comp_len': len(p[5]),
                'node_h': stable_hash_1000(p[4]),
                '__y__': is_anom,
                '__seq__': seq
            })

    df = pd.DataFrame(rows)

    if len(df) > CFG['MAX_BGL']:
        # stratified cap, kemudian kembali ke urutan asli
        selected = []
        for cls in [0,1]:
            g = df[df['__y__'] == cls]
            n = int(round(CFG['MAX_BGL'] * len(g) / len(df)))
            n = min(n, len(g))
            selected.append(g.sample(n=n, random_state=CFG['SEED']))
        df = pd.concat(selected).sort_values('__seq__').reset_index(drop=True)

    y = df.pop('__y__').astype(int).values
    df.drop(columns=['__seq__'], inplace=True)

    # PENTING: jangan drop_duplicates atas engineered features.
    X = df
    return X, y

In [ ]:
# ============================================================
# 4. UNSW-NB15
# ============================================================
UNSW_COLUMNS = [
    'srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
    'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
    'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
    'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
    'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
    'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
    'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
    'ct_dst_src_ltm','attack_cat','label'
]

def load_unsw_verify():
    # Pertama cek apakah file sudah memiliki header
    probe = pd.read_csv(CFG['UNSW'], nrows=3, low_memory=False)
    probe_cols = clean_cols(probe.columns)

    if 'label' in probe_cols or 'is_attack' in probe_cols:
        df = pd.read_csv(CFG['UNSW'], low_memory=False)
        df.columns = clean_cols(df.columns)
        print("UNSW format: header terdeteksi")
    else:
        raw = pd.read_csv(CFG['UNSW'], header=None, low_memory=False)
        if raw.shape[1] != len(UNSW_COLUMNS):
            raise ValueError(
                f"UNSW tanpa header memiliki {raw.shape[1]} kolom; "
                f"schema yang diharapkan {len(UNSW_COLUMNS)} kolom."
            )
        raw.columns = UNSW_COLUMNS
        df = raw
        print("UNSW format: tanpa header → schema 49 kolom diterapkan")

    label_col = 'label' if 'label' in df.columns else 'is_attack'
    y = (pd.to_numeric(df[label_col], errors='coerce').fillna(0) != 0).astype(int).values

    # Identifiers dan explicit target semantics tidak dipakai sebagai predictor.
    X = df.drop(columns=[
        label_col, 'attack_cat',
        'srcip','dstip','src_ip','dst_ip',
        'stime','ltime'
    ], errors='ignore').copy()

    X.columns = clean_cols(X.columns)

    # Di STEP 1 tidak encoding/modeling. Kita hanya verifikasi tipe fitur.
    return X, y

In [ ]:
# ============================================================
# 5. LOAD + AUDIT
# ============================================================
DATASETS = {}

print("\nLoading CICIDS2018...")
DATASETS['CICIDS2018'] = load_cicids_verify()

print("\nLoading HDFS...")
DATASETS['HDFS'] = load_hdfs_verify()

print("\nLoading BGL...")
DATASETS['BGL'] = load_bgl_verify()

print("\nLoading UNSW-NB15...")
DATASETS['UNSW-NB15'] = load_unsw_verify()

for name,(X,y) in DATASETS.items():
    audit_df(name, X, y)

In [ ]:
# ============================================================
# 6. FINAL GATE — JANGAN LANJUT KE S0 JIKA ADA YANG GAGAL
# ============================================================
summary = []
for name,(X,y) in DATASETS.items():
    summary.append({
        'dataset': name,
        'rows': len(X),
        'features': X.shape[1],
        'positive_n': int(np.sum(y)),
        'positive_pct': float(100*np.mean(y)),
        'duplicate_feature_vectors_audit_only': int(X.duplicated().sum())
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

print("\n=== STEP 1 STATUS ===")
if len(DATASETS) == 4 and all(len(v[0]) > 0 for v in DATASETS.values()):
    print("✅ Empat dataset berhasil dimuat.")
    print("STOP DI SINI. Kirim output tabel summary kepada ChatGPT sebelum menjalankan S0.")
else:
    print("❌ Belum semua dataset valid. Jangan lanjut ke S0.")